#1. Set up the environment

In [ ]:
#Step 1: import necessary libraries - done
!pip -q install google-api-python-client google-auth-httplib2 google-auth-oauthlib beautifulsoup4 gspread

#Step 2: authenticate in colab - get rid of this
from google.colab import auth
auth.authenticate_user()
print("Authenticated")

Authenticated


In [ ]:
#Step 3: Build drive API client - done
from googleapiclient.discovery import build
drive = build("drive", "v3")
print("Drive client ready")

Drive client ready


#2. Html processing

##2.1 Read sheets and extract google docs link for processing

In [3]:
#Step 4: Authenticate and build gspread client
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

#Step 5:scan sheets to get docs_url
SHEET_URL = "https://docs.google.com/spreadsheets/d/1sMgzVgxNzT5RcYHlrDlQkNKYkfwTWxiys1vYhNWF38Q/edit?gid=1454104928#gid=1454104928"
SHEET_TAB = "DSBV | Rackcosmo"

def is_blank(x):
    return (x is None) or (isinstance(x, str) and x.strip() == "")

def get_docs_urls_to_publish(sheet_url: str,worksheet_title: str,) -> list[str]:
    sh = gc.open_by_url(sheet_url)
    ws = sh.worksheet(worksheet_title) if worksheet_title else sh.sheet1

    # Get rows as dicts using header row; request UNFORMATTED so checkboxes are booleans
    rows = ws.get_all_records(value_render_option="UNFORMATTED_VALUE", head=1)

    urls = []
    for r in rows:
        approved = r.get("Duyệt đăng")
        status   = r.get("Trạng thái")
        link     = r.get("Link docs Bài viết")

        # Checkbox comes as True/False; guard for string fallbacks just in case
        approved_bool = (approved is True) or (str(approved).upper() == "TRUE")
        if approved_bool and is_blank(status) and link:
            urls.append(link)

    return urls
docs_urls = get_docs_urls_to_publish(SHEET_URL, SHEET_TAB)
print(f"Found {len(docs_urls)} docs ready to export.")
docs_urls[:5]

Found 1 docs ready to export.


['https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing']

##2.2 Turn docs into Html

In [4]:
#Step 6: Parse the file ID from any Google Docs URL
import re
from urllib.parse import urlparse, parse_qs
from bs4 import BeautifulSoup
import os

def extract_file_id(docs_urls: str) -> str:
    """
    Extracts the Google file ID from common Google Docs URLs.
    Supports:
      - https://docs.google.com/document/d/<ID>/edit
      - https://docs.google.com/document/d/<ID>/
      - https://docs.google.com/open?id=<ID>
    """
    # Try /document/d/<ID>/...
    m = re.search(r"/document/d/([a-zA-Z0-9-_]+)", docs_urls)
    if m:
        return m.group(1)

    # Fallback: open?id=<ID>
    parsed = urlparse(docs_urls)
    qs = parse_qs(parsed.query)
    if "id" in qs and qs["id"]:
        return qs["id"][0]

    raise ValueError("Could not extract file ID from the provided URL.")

#Step 7 — Export the Doc to HTML
def export_doc_to_html(docs_id: str) -> bytes:
    """
    Export a Google Doc to HTML using Drive API.
    Returns raw HTML bytes.
    """
    html_files = drive.files().export(
        fileId=docs_id,
        mimeType="text/html"
    ).execute()
    return html_files

def save_html(html_files: bytes, out_path: str) -> None:
    with open(out_path, "wb") as f:
        f.write(html_files)

# Step 8 - Upload the HTML file to Google Drive
def upload_to_drive(filepath: str, folder_id: str) -> None:
    """
    Uploads a file to a specific folder in Google Drive.
    """
    file_metadata = {
        'name': os.path.basename(filepath),
        'parents': [folder_id]
    }
    media = MediaFileUpload(filepath, mimetype='text/html')
    file = drive.files().create(body=file_metadata, media_body=media, fields='id').execute()
    print(f"File ID: {file.get('id')}")


# Initialize a counter for cumulative numbering
if 'file_counter' not in globals():
    file_counter = 0

# Keep track of filenames to handle duplicates and successfully uploaded files
if 'saved_filenames' not in globals():
    saved_filenames = {}
if 'uploaded_filenames' not in globals():
    uploaded_filenames = []

# Specify the Google Drive folder ID where you want to upload the files
# Replace 'YOUR_FOLDER_ID' with the actual ID of your Google Drive folder
DRIVE_FOLDER_ID = '1NjOhY5tEOFUrz49YeQiA_fwGzKg_4CJX'

# Execution for multiple URLs
if docs_urls:  # Check if the list of URLs is not empty
    for docs_url in docs_urls:
        global file_counter # Moved global declaration to the top of the loop
        global saved_filenames # Declare saved_filenames as global
        global uploaded_filenames # Declare uploaded_filenames as global

        try:
            docs_id = extract_file_id(docs_url)
            print(f"Processing doc ID: {docs_id} from URL: {docs_url}")

            # Export to HTML
            html_content_bytes = export_doc_to_html(docs_id)
            html_content_str = html_content_bytes.decode('utf-8') # Decode bytes to string

            # Parse the HTML content with BeautifulSoup to find the first <h1> tag
            soup = BeautifulSoup(html_content_str, 'html.parser')
            h1_tag = soup.find('h1')

            # Determine the filename based on the h1 tag
            base_filename = ""
            if h1_tag and h1_tag.get_text(strip=True):
                # Sanitize the text to create a valid filename
                base_filename = re.sub(r'[^\w\s-]', '', h1_tag.get_text(strip=True)).strip()
                base_filename = re.sub(r'[-\s]+', '-', base_filename)

            if not base_filename:
                # Fallback if h1 is empty after sanitization or not found
                base_filename = f"exported_doc"

            # Handle duplicate filenames by appending a counter
            filename_prefix = base_filename
            counter = 1
            output_filename = f"{filename_prefix}.html"
            while output_filename in saved_filenames:
                counter += 1
                output_filename = f"{filename_prefix}_{counter}.html"

            # Add the filename to the set of saved filenames
            saved_filenames[output_filename] = True

            # Increment the counter and specify the output file path with the determined name
            file_counter += 1
            output_path = output_filename # Use the generated filename

            # Save the HTML content to the file
            save_html(html_content_bytes, output_path) # save_html expects bytes
            print(f"HTML content saved to {output_path}")

            # Upload the saved HTML file to Google Drive
            from googleapiclient.http import MediaFileUpload
            upload_to_drive(output_path, DRIVE_FOLDER_ID)
            print(f"HTML content uploaded to Google Drive folder: {DRIVE_FOLDER_ID}")
            uploaded_filenames.append(output_path) # Add successfully uploaded filename to the list


        except ValueError as e:
            print(f"Error processing URL {docs_url}: {e}")
        except Exception as e:
            print(f"An unexpected error occurred while processing {docs_url}: {e}")
else:
    print("No document URLs found to process.")

Processing doc ID: 19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E from URL: https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing
HTML content saved to Kệ-trung-tải-4-tầng-Tối-ưu-hóa-không-gian-theo-chiều-cao-tăng-gấp-đôi-hiệu-suất.html
File ID: 1vPuxdvZqFeOw3rAQYTeY4iYSfCZxXudE
HTML content uploaded to Google Drive folder: 1NjOhY5tEOFUrz49YeQiA_fwGzKg_4CJX


##2.3 Processing images

###2.3.1 Extract all image tags

In [5]:
# Step 9: Process each HTML file and extract image tags
from bs4 import BeautifulSoup

def extract_images_from_html(filepath: str) -> list:
    """
    Reads an HTML file, parses it, and extracts all <img> tags.
    Returns a list of dictionaries, where each dictionary represents an <img> tag's attributes.
    """
    images = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            html_content = f.read()

        soup = BeautifulSoup(html_content, 'html.parser')
        img_tags = soup.find_all('img')

        for img in img_tags:
            images.append(img.attrs) # Get all attributes of the img tag

    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
    except Exception as e:
        print(f"An error occurred while processing {filepath}: {e}")

    return images

# Process each successfully uploaded HTML file
all_images = {}
for filename in uploaded_filenames: # Iterate through the list of uploaded filenames
    filepath = filename # Files were saved in the current directory
    images_in_file = extract_images_from_html(filepath)
    all_images[filename] = images_in_file

print(f"Extracted images from {len(all_images)} files.")

# Display the extracted image attributes (optional)
# for filename, images in all_images.items():
#     print(f"\nImages found in {filename}:")
#     if images:
#         for img_attrs in images:
#             print(img_attrs)
#     else:
#         print("No images found.")

Extracted images from 1 files.


In [6]:
# Get the filename of the first uploaded document
first_uploaded_filename = uploaded_filenames[0]

# Get the images for the first document
first_doc_images = all_images[first_uploaded_filename]

# Print the extracted images for the first document
print(f"Images found in the first document ({first_uploaded_filename}):")
if first_doc_images:
    for img_attrs in first_doc_images:
        print(img_attrs)
else:
    print("No images found in the first document.")

Images found in the first document (Kệ-trung-tải-4-tầng-Tối-ưu-hóa-không-gian-theo-chiều-cao-tăng-gấp-đôi-hiệu-suất.html):
{'alt': 'Kệ trung tải 4 tầng', 'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXcuP4ohYa6hWXz8ElzVkbJHmv8IYPkzsz5MqRFpbxoJdOx5GD8B7_5Whcud-b3C9qe5xxPtKRry_O27HcVcKXOBxVSROm_tY8GIV1xhyEvuPIpOVoIxSo5TEkoBl2E6Iuk7djo3PA?key=5axpOQhltCUHo8WHcYAGCg', 'style': 'width: 601.70px; height: 450.67px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translateZ(0px);', 'title': ''}
{'alt': 'Bản vẽ chi tiết kệ', 'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXf-6bE5CggSrHHXr5CVN_CkH5knRLKd2FPuifRz_6B14wJfAVqVuRfWzc1x_pEPwqLX96RDX_82zw9iTR6QQgXx2xF4tYhVEQ8XqepE3Zj8jPrwEZBKSxkh1-XmQD1LvoBdIT37rg?key=5axpOQhltCUHo8WHcYAGCg', 'style': 'width: 601.70px; height: 450.67px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translat

###2.3.2 Upload images extracted to WordPress.

In [7]:
# Step 10: Install the requests library
!pip install requests

#### Authenticate with wordpress rest api


In [8]:
# Step 11: Set up authentication for the REST API using application passwords.
import base64

# Define your WordPress site URL and the application password
# Replace with your actual WordPress credentials
wordpress_rest_url = "https://ngoncareer.com/wp-json/wp/v2"  # Replace with your WordPress site URL + REST API endpoint
wordpress_username = "admin_career" # Replace with your WordPress username
wordpress_password = "efkV mie6 u3P8 C3mC NyM8 Ickp" # Replace with your WordPress application password

# Combine username and password for Basic Authentication
credentials = f"{wordpress_username}:{wordpress_password}"
encoded_credentials = base64.b64encode(credentials.encode()).decode()

# Construct the authentication header
auth_header = {
    "Authorization": f"Basic {encoded_credentials}"
}

print("WordPress REST API authentication header created.")

WordPress REST API authentication header created.


#### Prepare image filenames and slugs

In [9]:
# Step 12a: Prepare image filenames and slugs based on alt text
import slugify
import os
from urllib.parse import urlparse

prepared_images_data = {}

print("Preparing image filenames and slugs...")

# Process only the images from the most recently uploaded HTML file
if uploaded_filenames:
    latest_uploaded_filename = uploaded_filenames[0]
    images = all_images.get(latest_uploaded_filename, []) # Get images for the latest file

    if not images:
        print(f"No images found in {latest_uploaded_filename} to prepare.")
    else:
        prepared_images_data[latest_uploaded_filename] = []
        print(f"Processing images from {latest_uploaded_filename}:")

        for i, img_attrs in enumerate(images):
            image_url = img_attrs.get('src')
            if not image_url:
                print(f"  Skipping image with missing 'src' attribute in {latest_uploaded_filename}.")
                continue

            alt_text = img_attrs.get('alt', '')
            base_filename = ""

            if alt_text:
                # Take the first 5 words and create a slug
                alt_words = alt_text.split()[:5]
                base_filename = slugify.slugify(" ".join(alt_words))
                if not base_filename: # Fallback if slugify results in an empty string
                    base_filename = f"image_{os.path.splitext(latest_uploaded_filename)[0]}_{i+1}"
            else:
                 # Fallback if alt text is empty
                 base_filename = f"image_{os.path.splitext(latest_uploaded_filename)[0]}_{i+1}"

            # Try to get the extension from the original URL
            extension = os.path.splitext(urlparse(image_url).path)[1]
            if not extension:
                # Basic mapping for common image types if extension not in URL
                # This might need refinement or fetching content-type during download
                # For now, a simple guess or default
                if 'png' in image_url.lower():
                     extension = '.png'
                elif 'gif' in image_url.lower():
                     extension = '.gif'
                else:
                     extension = '.jpg' # Default to jpg if no extension found

            image_filename = f"{base_filename}{extension}"
            image_slug = base_filename # Use the base filename as a simple slug

            prepared_images_data[latest_uploaded_filename].append({
                'original_url': image_url,
                'filename': image_filename,
                'slug': image_slug,
                'alt_text': alt_text # Keep original alt text as well
            })

print("Finished preparing image filenames and slugs.")

# Display prepared data structure (optional)
# import json
# print(json.dumps(prepared_images_data, indent=2))

Preparing image filenames and slugs...
Processing images from Kệ-trung-tải-4-tầng-Tối-ưu-hóa-không-gian-theo-chiều-cao-tăng-gấp-đôi-hiệu-suất.html:
Finished preparing image filenames and slugs.


####Define resizing parameters




In [10]:
# Step 17: Set variables for the desired image width and height
target_image_width = 800  # Example width in pixels
target_image_height = 600 # Example height in pixels

print(f"Target image width set to: {target_image_width} pixels")
print(f"Target image height set to: {target_image_height} pixels")

Target image width set to: 800 pixels
Target image height set to: 600 pixels


#### Download images


In [11]:
# Step 18: Download original images
import requests
import os

downloaded_image_paths = {}
download_errors = []

print("\n--- Downloading Original Images ---")

if not prepared_images_data:
    print("No prepared image data found to download.")
else:
    # Iterate through the prepared image data structure, processing only the latest file
    for filename, images_data in prepared_images_data.items():
        print(f"\nProcessing images from {filename} for download:")
        if not images_data:
            print("No prepared image data found for this file.")
            continue

        for img_info in images_data:
            image_url = img_info['original_url']
            image_filename = img_info['filename']
            local_filepath = image_filename # Save in the current directory

            try:
                print(f"  Downloading image from: {image_url} to {local_filepath}")
                response = requests.get(image_url, stream=True)
                response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

                with open(local_filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)

                downloaded_image_paths[image_url] = local_filepath
                print(f"  Successfully downloaded image to {local_filepath}")

            except requests.exceptions.RequestException as e:
                download_errors.append({'original_url': image_url, 'filename': image_filename, 'error': str(e)})
                print(f"  Error downloading image from {image_url}: {e}")
            except Exception as e:
                download_errors.append({'original_url': image_url, 'filename': image_filename, 'error': str(e)})
                print(f"  An unexpected error occurred while downloading {image_url}: {e}")
            finally:
                if 'response' in locals() and response:
                    response.close() # Ensure the stream is closed


print("\n--- Image Download Summary ---")
print(f"Total images attempted: {sum(len(images_data) for images_data in prepared_images_data.values()) if prepared_images_data else 0}")
print(f"Successfully downloaded: {len(downloaded_image_paths)}")
print(f"Failed downloads: {len(download_errors)}")
if download_errors:
    print("Failed download details:")
    for error in download_errors:
        print(f"- Original URL: {error['original_url']}, Filename: {error['filename']}, Error: {error['error']}")

print("\n--- Downloaded Image Paths ---")
if downloaded_image_paths:
    # import json # Uncomment if you want to pretty print with json
    # print(json.dumps(downloaded_image_paths, indent=2))
    for original_url, local_path in downloaded_image_paths.items():
        print(f"Original: {original_url} -> Local: {local_path}")
else:
    print("No images were successfully downloaded.")


--- Downloading Original Images ---

Processing images from Kệ-trung-tải-4-tầng-Tối-ưu-hóa-không-gian-theo-chiều-cao-tăng-gấp-đôi-hiệu-suất.html for download:
  Successfully downloaded image to ke-trung-tai-4-tang.jpg
  Successfully downloaded image to ban-ve-chi-tiet-ke.jpg
  Successfully downloaded image to thiet-ke-ke-4-tang.jpg
  Successfully downloaded image to uu-diem-ke-4-tang.jpg
  Successfully downloaded image to ung-dung-ke-4-tang.jpg
  Successfully downloaded image to ke-4-tang-rackcosmo.jpg

--- Image Download Summary ---
Total images attempted: 6
Successfully downloaded: 6
Failed downloads: 0

--- Downloaded Image Paths ---
Original: https://lh7-rt.googleusercontent.com/docsz/AD_4nXcuP4ohYa6hWXz8ElzVkbJHmv8IYPkzsz5MqRFpbxoJdOx5GD8B7_5Whcud-b3C9qe5xxPtKRry_O27HcVcKXOBxVSROm_tY8GIV1xhyEvuPIpOVoIxSo5TEkoBl2E6Iuk7djo3PA?key=5axpOQhltCUHo8WHcYAGCg -> Local: ke-trung-tai-4-tang.jpg
Original: https://lh7-rt.googleusercontent.com/docsz/AD_4nXf-6bE5CggSrHHXr5CVN_CkH5knRLKd2FPuifRz

##### Resize images

### Subtask:
Use an image processing library (like Pillow) to resize the downloaded images to the specified dimensions.


**Reasoning**:
Use Pillow to resize the downloaded images to the target dimensions.



In [12]:
# Step 19: Resize downloaded images using Pillow
from PIL import Image

print("\n--- Resizing Downloaded Images ---")

if not downloaded_image_paths:
    print("No downloaded image paths available to resize.")
else:
    resized_image_count = 0
    failed_image_resizes = []

    for original_url, local_filepath in downloaded_image_paths.items():
        try:
            print(f"  Attempting to resize {local_filepath} to {target_image_width}x{target_image_height}")

            # Open the image file
            with Image.open(local_filepath) as img:
                # Resize the image
                resized_img = img.resize((target_image_width, target_image_height))

                # Save the resized image, overwriting the original file
                # Determine the format based on the original file extension
                file_extension = os.path.splitext(local_filepath)[1].lower()
                save_format = resized_img.format if resized_img.format else 'PNG' # Default to PNG if format is None

                # Handle specific formats like JPEG which don't support alpha channel when saving
                if save_format == 'JPEG' and resized_img.mode == 'RGBA':
                    resized_img = resized_img.convert('RGB')
                    save_format = 'JPEG' # Ensure format is explicitly set after conversion

                resized_img.save(local_filepath, format=save_format)

            resized_image_count += 1
            print(f"  Successfully resized and saved {local_filepath}")

        except FileNotFoundError:
            failed_image_resizes.append({'filepath': local_filepath, 'error': 'File not found'})
            print(f"  Error: File not found for resizing at {local_filepath}")
        except IOError as e:
            failed_image_resizes.append({'filepath': local_filepath, 'error': str(e)})
            print(f"  IOError resizing {local_filepath}: {e}")
        except Exception as e:
            failed_image_resizes.append({'filepath': local_filepath, 'error': str(e)})
            print(f"  An unexpected error occurred while resizing {local_filepath}: {e}")


print("\n--- Image Resizing Summary ---")
print(f"Total images attempted to resize: {len(downloaded_image_paths)}")
print(f"Successfully resized: {resized_image_count}")
print(f"Failed resizes: {len(failed_image_resizes)}")
if failed_image_resizes:
    print("Failed image resize details:")
    for failure in failed_image_resizes:
        print(f"- Filepath: {failure['filepath']}, Error: {failure['error']}")



--- Resizing Downloaded Images ---
  Attempting to resize ke-trung-tai-4-tang.jpg to 800x600
  Successfully resized and saved ke-trung-tai-4-tang.jpg
  Attempting to resize ban-ve-chi-tiet-ke.jpg to 800x600
  Successfully resized and saved ban-ve-chi-tiet-ke.jpg
  Attempting to resize thiet-ke-ke-4-tang.jpg to 800x600
  Successfully resized and saved thiet-ke-ke-4-tang.jpg
  Attempting to resize uu-diem-ke-4-tang.jpg to 800x600
  Successfully resized and saved uu-diem-ke-4-tang.jpg
  Attempting to resize ung-dung-ke-4-tang.jpg to 800x600
  Successfully resized and saved ung-dung-ke-4-tang.jpg
  Attempting to resize ke-4-tang-rackcosmo.jpg to 800x600
  Successfully resized and saved ke-4-tang-rackcosmo.jpg

--- Image Resizing Summary ---
Total images attempted to resize: 6
Successfully resized: 6
Failed resizes: 0


#### Modify image upload


In [13]:
# Step 12b (Revised): Iterate through the resized images and upload them to WordPress using the REST API's media endpoint.
import requests
import os
from mimetypes import guess_type

uploaded_image_count = 0
failed_image_uploads = []
uploaded_image_mapping = {} # To store mapping of original URL to new WordPress URL
uploaded_media_data_list = [] # Initialize a list to store uploaded media data

# Ensure the media endpoint URL is correct
media_endpoint = f"{wordpress_rest_url}/media"

print(f"Starting image upload from local files to WordPress media endpoint: {media_endpoint}")

# Iterate through the prepared image data structure, processing only the latest file
if prepared_images_data:
    # Assuming prepared_images_data will only contain the data for the latest file after the previous step
    for filename, images_data in prepared_images_data.items():
        print(f"\nProcessing images from {filename} for upload:")
        if not images_data:
            print("No prepared image data found for this file.")
            continue

        for img_info in images_data:
            image_url = img_info['original_url']
            image_filename = img_info['filename']
            image_slug = img_info['slug']
            alt_text = img_info['alt_text']

            # Get the local filepath from the downloaded_image_paths mapping
            local_filepath = downloaded_image_paths.get(image_url)

            if not local_filepath or not os.path.exists(local_filepath):
                failed_image_uploads.append({'original_url': image_url, 'filename': image_filename, 'status_code': 'Error', 'response': 'Local file not found'})
                print(f"  Error: Local file not found for original URL {image_url}. Skipping upload.")
                continue

            try:
                # Read the image content from the local file
                print(f"  Attempting to read image from local file: {local_filepath}")
                with open(local_filepath, 'rb') as f:
                    image_content = f.read()

                # Determine the mime type from the local file
                mime_type, _ = guess_type(local_filepath)
                if not mime_type:
                    mime_type = 'application/octet-stream' # Final fallback

                # Prepare data for WordPress REST API upload
                files = {
                    'file': (image_filename, image_content, mime_type)
                }

                # Include slug, alt_text, caption, and description in the media data
                media_data = {
                    'title': image_slug, # Use slug as title for media item
                    'alt_text': alt_text, # Include original alt text
                    'caption': alt_text, # Using alt_text for caption
                    'description': alt_text, # Using alt_text for description
                }


                # Send POST request to upload the image
                print(f"  Uploading {image_filename} to WordPress...")
                upload_response = requests.post(media_endpoint, headers=auth_header, files=files, data=media_data)


                if upload_response.status_code in [200, 201]: # 200 OK or 201 Created
                    uploaded_image_count += 1
                    uploaded_media_data = upload_response.json()
                    wp_media_id = uploaded_media_data.get('id')
                    wp_media_url = uploaded_media_data.get('source_url')

                    print(f"  Successfully uploaded {image_filename}. WordPress Media ID: {wp_media_id}, URL: {wp_media_url}")

                    # Store the mapping of the original URL to the new WordPress URL
                    uploaded_image_mapping[image_url] = wp_media_url

                    # Append the uploaded media data to the list
                    uploaded_media_data_list.append(uploaded_media_data)

                else:
                    failed_image_uploads.append({'original_url': image_url, 'filename': image_filename, 'status_code': upload_response.status_code, 'response': upload_response.text})
                    print(f"  Failed to upload {image_filename}. Status Code: {upload_response.status_code}, Response: {upload_response.text}")

            except requests.exceptions.RequestException as e:
                failed_image_uploads.append({'original_url': image_url, 'filename': image_filename, 'status_code': 'Error', 'response': str(e)})
                print(f"  Error uploading image from {local_filepath}: {e}")
            except Exception as e:
                failed_image_uploads.append({'original_url': image_url, 'filename': image_filename, 'status_code': 'Error', 'response': str(e)})
                print(f"  An unexpected error occurred while processing {local_filepath}: {e}")


print("\n--- Image Upload Summary ---")
print(f"Total images attempted: {sum(len(images_data) for images_data in prepared_images_data.values()) if prepared_images_data else 0}")
print(f"Successfully uploaded: {uploaded_image_count}")
print(f"Failed uploads: {len(failed_image_uploads)}")
if failed_image_uploads:
    print("Failed image details:")
    for failure in failed_image_uploads:
        print(f"- Original URL: {failure['original_url']}, Filename: {failure['filename']}, Status: {failure['status_code']}, Response: {failure['response']}")

print("\n--- Uploaded Image URL Mapping ---")
if uploaded_image_mapping:
    # import json # Uncomment if you want to pretty print with json
    # print(json.dumps(uploaded_image_mapping, indent=2))
    for original_url, wp_url in uploaded_image_mapping.items():
        print(f"Original: {original_url} -> WordPress: {wp_url}")
else:
    print("No images were successfully uploaded or mapped.")

print("\n--- New Image Links ---")
if uploaded_image_mapping:
    new_image_links = list(uploaded_image_mapping.values())
    for link in new_image_links:
        print(link)
else:
    print("No new image links to display.")

Starting image upload from local files to WordPress media endpoint: https://ngoncareer.com/wp-json/wp/v2/media

Processing images from Kệ-trung-tải-4-tầng-Tối-ưu-hóa-không-gian-theo-chiều-cao-tăng-gấp-đôi-hiệu-suất.html for upload:
  Attempting to read image from local file: ke-trung-tai-4-tang.jpg
  Uploading ke-trung-tai-4-tang.jpg to WordPress...
  Successfully uploaded ke-trung-tai-4-tang.jpg. WordPress Media ID: 47147, URL: https://ngoncareer.com/wp-content/uploads/2025/09/ke-trung-tai-4-tang-17.png
  Attempting to read image from local file: ban-ve-chi-tiet-ke.jpg
  Uploading ban-ve-chi-tiet-ke.jpg to WordPress...
  Successfully uploaded ban-ve-chi-tiet-ke.jpg. WordPress Media ID: 47148, URL: https://ngoncareer.com/wp-content/uploads/2025/09/ban-ve-chi-tiet-ke-17.png
  Attempting to read image from local file: thiet-ke-ke-4-tang.jpg
  Uploading thiet-ke-ke-4-tang.jpg to WordPress...
  Successfully uploaded thiet-ke-ke-4-tang.jpg. WordPress Media ID: 47149, URL: https://ngoncareer

#### Clean up temporary files



In [14]:
# Step 20: Remove locally saved resized images
import os

print("\n--- Cleaning up locally saved images ---")

removed_count = 0
failed_removals = []

if not downloaded_image_paths:
    print("No downloaded image paths available for cleanup.")
else:
    for original_url, local_filepath in downloaded_image_paths.items():
        if os.path.exists(local_filepath):
            try:
                print(f"  Removing local file: {local_filepath}")
                os.remove(local_filepath)
                removed_count += 1
                print(f"  Successfully removed {local_filepath}")
            except OSError as e:
                failed_removals.append({'filepath': local_filepath, 'error': str(e)})
                print(f"  Error removing file {local_filepath}: {e}")
            except Exception as e:
                failed_removals.append({'filepath': local_filepath, 'error': str(e)})
                print(f"  An unexpected error occurred while removing {local_filepath}: {e}")
        else:
            print(f"  File not found during cleanup: {local_filepath}")


print("\n--- Local Image Cleanup Summary ---")
print(f"Total files attempted to remove: {len(downloaded_image_paths)}")
print(f"Successfully removed: {removed_count}")
print(f"Failed removals: {len(failed_removals)}")
if failed_removals:
    print("Failed removal details:")
    for failure in failed_removals:
        print(f"- Filepath: {failure['filepath']}, Error: {failure['error']}")


--- Cleaning up locally saved images ---
  Removing local file: ke-trung-tai-4-tang.jpg
  Successfully removed ke-trung-tai-4-tang.jpg
  Removing local file: ban-ve-chi-tiet-ke.jpg
  Successfully removed ban-ve-chi-tiet-ke.jpg
  Removing local file: thiet-ke-ke-4-tang.jpg
  Successfully removed thiet-ke-ke-4-tang.jpg
  Removing local file: uu-diem-ke-4-tang.jpg
  Successfully removed uu-diem-ke-4-tang.jpg
  Removing local file: ung-dung-ke-4-tang.jpg
  Successfully removed ung-dung-ke-4-tang.jpg
  Removing local file: ke-4-tang-rackcosmo.jpg
  Successfully removed ke-4-tang-rackcosmo.jpg

--- Local Image Cleanup Summary ---
Total files attempted to remove: 6
Successfully removed: 6
Failed removals: 0


#3. Create empty posts in wordpress

## 3.1. Extract heading 1 of docs

In [15]:
# Step 13: Read the ORIGINAL HTML file and extract the heading
from bs4 import BeautifulSoup
import os

def extract_heading_from_html(html_content: str) -> str:
    """
    Parses HTML content and extracts the text content of the first <h1> tag.
    Returns the heading text or an empty string if no <h1> tag is found.
    """
    heading = ""
    try:
        soup = BeautifulSoup(html_content, 'html.parser')
        h1_tag = soup.find('h1')

        if h1_tag and h1_tag.get_text(strip=True):
            heading = h1_tag.get_text(strip=True)

    except Exception as e:
        print(f"An error occurred while parsing HTML content: {e}")

    return heading

# Assuming the original HTML file is saved with a name based on the first docs_url
# We need to get the filename that was used to save the original HTML
# This filename is stored in the uploaded_filenames list (before cleanup)

h1_heading = ""
if uploaded_filenames:
    # Get the filename of the original HTML file saved in Step 7
    original_html_filepath = uploaded_filenames[0] # Assuming only one doc is processed at a time

    if os.path.exists(original_html_filepath):
        try:
            print(f"Reading original HTML file: {original_html_filepath}")
            with open(original_html_filepath, 'r', encoding='utf-8') as f:
                original_html_content = f.read()

            h1_heading = extract_heading_from_html(original_html_content)

            if h1_heading:
                print(f"Extracted heading from original HTML: {h1_heading}")
            else:
                print(f"No heading found in original HTML file.")

        except FileNotFoundError:
             print(f"Error: Original HTML file not found at {original_html_filepath}")
        except Exception as e:
            print(f"An unexpected error occurred while reading or parsing {original_html_filepath}: {e}")

    else:
        print(f"Original HTML file not found at {original_html_filepath}. It might have been deleted.")
else:
    print("No uploaded filenames available. Cannot determine original HTML file.")

# You can now use the 'h1_heading' variable for later steps

Reading original HTML file: Kệ-trung-tải-4-tầng-Tối-ưu-hóa-không-gian-theo-chiều-cao-tăng-gấp-đôi-hiệu-suất.html
Extracted heading from original HTML: Kệ trung tải 4 tầng: Tối ưu hóa không gian theo chiều cao, tăng gấp đôi hiệu suất


## 3.2. Slugify the main keyword

In [16]:
# Step 14: Get the main keyword from the Google Sheet and slugify it
import slugify

# Re-open the sheet to get the latest data
sh = gc.open_by_url(SHEET_URL)
ws = sh.worksheet(SHEET_TAB) if SHEET_TAB else sh.sheet1

# Get rows as dicts using header row
rows = ws.get_all_records(head=1)

main_keyword = ""
# Assuming we are processing the first document found in docs_urls,
# we need to find the corresponding row in the sheet to get the main keyword.
# A more robust approach might involve matching by the document URL or ID if available in the sheet.
# For now, we'll assume the first row in 'rows' corresponds to the first URL in 'docs_urls'
# if docs_urls is not empty and rows is not empty.
if docs_urls and rows:
    # Find the row corresponding to the first processed document URL
    # This is a simplified assumption and might need adjustment based on sheet structure
    target_row = None
    for r in rows:
        link = r.get("Link docs Bài viết")
        if link == docs_urls[0]:
            target_row = r
            break

    if target_row:
        main_keyword = target_row.get("main keyword", "") # Get the value from the "Main Keyword" column

if main_keyword:
    slugified_main_keyword = slugify.slugify(main_keyword)
    print(f"Extracted Main Keyword: {main_keyword}")
    print(f"Slugified Main Keyword: {slugified_main_keyword}")
else:
    print("Could not find Main Keyword in the Google Sheet for the processed document.")

# You can now use the 'slugified_main_keyword' variable for later steps

Extracted Main Keyword: kệ trung tải 4 tầng
Slugified Main Keyword: ke-trung-tai-4-tang


## 3.3. Create emtpy post in wordpress

In [17]:
# Step 15: Create an empty post in WordPress
import requests

# Ensure the posts endpoint URL is correct
posts_endpoint = f"{wordpress_rest_url}/posts"

print(f"Attempting to create a new post in WordPress at: {posts_endpoint}")

# Prepare the data for the new post
# Set status to 'draft' to create an empty draft post
post_data = {
    'title': h1_heading, # Use the extracted H1 heading as the post title
    'slug': slugified_main_keyword, # Use the slugified main keyword as the post slug
    'status': 'draft',  # Create the post as a draft
    'content': '', # Start with empty content, we'll add HTML later
}

try:
    # Send POST request to create the post
    create_post_response = requests.post(posts_endpoint, headers=auth_header, json=post_data)

    if create_post_response.status_code in [200, 201]: # 200 OK or 201 Created
        created_post_data = create_post_response.json()
        new_post_id = created_post_data.get('id')
        new_post_link = created_post_data.get('link')
        print(f"Successfully created draft post. WordPress Post ID: {new_post_id}, Link: {new_post_link}")

        # Store the new post ID for later use (e.g., updating content, setting featured image)
        # Make sure wordpress_post_id is a global variable if needed in other cells
        global wordpress_post_id
        wordpress_post_id = new_post_id
        print(f"Stored new WordPress Post ID: {wordpress_post_id}")

    else:
        print(f"Failed to create draft post. Status Code: {create_post_response.status_code}, Response: {create_post_response.text}")

except requests.exceptions.RequestException as e:
    print(f"Error creating draft post: {e}")
except Exception as e:
    print(f"An unexpected error occurred while creating the draft post: {e}")

Attempting to create a new post in WordPress at: https://ngoncareer.com/wp-json/wp/v2/posts
Successfully created draft post. WordPress Post ID: 47153, Link: https://ngoncareer.com/?p=47153
Stored new WordPress Post ID: 47153


#4. HTML processing

##4.1 Update metadata of images

In [18]:
# Step 16: Update uploaded images with metadata and associate with the post
import requests

print("\n--- Updating Image Metadata and Associating with Post ---")

# Check if wordpress_post_id is defined and not None
if 'wordpress_post_id' not in globals() or wordpress_post_id is None:
     print("WordPress Post ID is not available. Cannot associate images with a post.")
elif not uploaded_image_mapping or not uploaded_media_data_list:
    print("Image upload data is incomplete. Cannot update images.")
else:
    updated_image_count = 0
    failed_image_updates = []

    # Ensure the media endpoint URL is correct for updating a specific item
    media_update_endpoint_base = f"{wordpress_rest_url}/media"

    # Iterate through the successfully uploaded image mapping
    for original_url, wp_url in uploaded_image_mapping.items():
        # Find the corresponding image data in prepared_images_data to get slug and alt_text
        img_info = None
        # Assuming prepared_images_data only contains data for the latest file
        if prepared_images_data:
            latest_filename = list(prepared_images_data.keys())[0]
            for info in prepared_images_data.get(latest_filename, []):
                if info['original_url'] == original_url:
                    img_info = info
                    break

        if img_info:
            image_slug = img_info['slug']
            alt_text = img_info['alt_text']

            wp_media_id = None
            # Find the media ID corresponding to the wp_url
            if 'uploaded_media_data_list' in globals(): # Check if the list exists from the upload step
                for media_item in uploaded_media_data_list:
                    if media_item.get('source_url') == wp_url:
                        wp_media_id = media_item.get('id')
                        break

            if wp_media_id:
                media_update_endpoint = f"{media_update_endpoint_base}/{wp_media_id}"
                print(f"  Attempting to update media item ID: {wp_media_id}")

                # Prepare data for the update request
                update_data = {
                    'slug': image_slug,
                    'alt_text': alt_text,
                    'caption': alt_text, # Using alt_text for caption as requested
                    'description': alt_text, # Using alt_text for description as requested
                    'post': wordpress_post_id # Associate with the newly created post
                }

                try:
                    # Send PUT request to update the media item
                    update_response = requests.put(media_update_endpoint, headers=auth_header, json=update_data)

                    if update_response.status_code in [200]: # 200 OK for update
                        updated_image_count += 1
                        print(f"  Successfully updated media item ID: {wp_media_id}")
                    else:
                        failed_image_updates.append({'wp_media_id': wp_media_id, 'status_code': update_response.status_code, 'response': update_response.text})
                        print(f"  Failed to update media item ID: {wp_media_id}. Status Code: {update_response.status_code}, Response: {update_response.text}")

                except requests.exceptions.RequestException as e:
                    failed_image_updates.append({'wp_media_id': wp_media_id, 'status_code': 'Error', 'response': str(e)})
                    print(f"  Error updating media item ID {wp_media_id}: {e}")
                except Exception as e:
                    failed_image_updates.append({'wp_media_id': wp_media_id, 'status_code': 'Error', 'response': str(e)})
                    print(f"  An unexpected error occurred while updating media item ID {wp_media_id}: {e}")
            else:
                 print(f"  Could not find WordPress Media ID for URL: {wp_url}. Skipping update.")
        else:
            print(f"  Could not find prepared image data for original URL: {original_url}. Skipping update.")


    print("\n--- Image Update Summary ---")
    print(f"Successfully updated: {updated_image_count}")
    print(f"Failed updates: {len(failed_image_updates)}")
    if failed_image_updates:
        print("Failed image update details:")
        for failure in failed_image_updates:
            print(f"- Media ID: {failure.get('wp_media_id', 'N/A')}, Status: {failure['status_code']}, Response: {failure['response']}")

    print("\n--- Uploaded Image URL Mapping ---")
    if uploaded_image_mapping:
        # import json # Uncomment if you want to pretty print with json
        # print(json.dumps(uploaded_image_mapping, indent=2))
        for original_url, wp_url in uploaded_image_mapping.items():
            print(f"Original: {original_url} -> WordPress: {wp_url}")
    else:
        print("No images were successfully uploaded or mapped.")

    print("\n--- New Image Links ---")
    if uploaded_image_mapping:
        new_image_links = list(uploaded_image_mapping.values())
        for link in new_image_links:
            print(link)
    else:
        print("No new image links to display.")


--- Updating Image Metadata and Associating with Post ---
  Attempting to update media item ID: 47147
  Successfully updated media item ID: 47147
  Attempting to update media item ID: 47148
  Successfully updated media item ID: 47148
  Attempting to update media item ID: 47149
  Successfully updated media item ID: 47149
  Attempting to update media item ID: 47150
  Successfully updated media item ID: 47150
  Attempting to update media item ID: 47151
  Successfully updated media item ID: 47151
  Attempting to update media item ID: 47152
  Successfully updated media item ID: 47152

--- Image Update Summary ---
Successfully updated: 6
Failed updates: 0

--- Uploaded Image URL Mapping ---
Original: https://lh7-rt.googleusercontent.com/docsz/AD_4nXcuP4ohYa6hWXz8ElzVkbJHmv8IYPkzsz5MqRFpbxoJdOx5GD8B7_5Whcud-b3C9qe5xxPtKRry_O27HcVcKXOBxVSROm_tY8GIV1xhyEvuPIpOVoIxSo5TEkoBl2E6Iuk7djo3PA?key=5axpOQhltCUHo8WHcYAGCg -> WordPress: https://ngoncareer.com/wp-content/uploads/2025/09/ke-trung-tai-4-tan

##4.2 Process Html input together with images metadata and extracted <h1>

### Image processing

In [19]:
# Step 17: Process the HTML file and replace image URLs and attributes
from bs4 import BeautifulSoup

print("\n--- Processing HTML and Replacing Image Tags ---")

if not uploaded_filenames:
    print("No HTML files were exported to process.")
elif not uploaded_image_mapping or not uploaded_media_data_list:
    print("Image upload data is incomplete. Cannot replace image tags.")
else:
    # Assuming we are processing the HTML file corresponding to the first uploaded filename
    html_filepath = uploaded_filenames[0]
    processed_html_content = ""
    images_replaced_count = 0

    try:
        print(f"  Reading HTML file: {html_filepath}")
        with open(html_filepath, 'r', encoding='utf-8') as f:
            html_content = f.read()

        soup = BeautifulSoup(html_content, 'html.parser')
        img_tags = soup.find_all('img')

        print(f"  Found {len(img_tags)} image tags in the HTML.")

        # Create a dictionary for quick lookup of uploaded media data by original URL
        uploaded_media_lookup = {}
        for original_url, wp_url in uploaded_image_mapping.items():
             # Find the corresponding media data in the list
             for media_item in uploaded_media_data_list:
                 if media_item.get('source_url') == wp_url:
                     uploaded_media_lookup[original_url] = media_item
                     break


        for img_tag in img_tags:
            original_src = img_tag.get('src')

            if original_src and original_src in uploaded_media_lookup:
                media_data = uploaded_media_lookup[original_src]
                new_src = media_data.get('source_url')
                new_alt_text = media_data.get('alt_text', '')
                media_details = media_data.get('media_details', {})
                new_width = media_details.get('width')
                new_height = media_details.get('height')
                wp_media_id = media_data.get('id')

                # Form the updated image tag string
                class_attr = f'aligncenter size-full wp-image-{wp_media_id}' if wp_media_id else ''
                updatedImgTag = f'<img class="{class_attr}" title="" src="{new_src}" alt="{new_alt_text}" width="{new_width}" height="{new_height}" />'

                # Form the WordPress caption shortcode string
                caption_shortcode = f'[caption id="attachment_{wp_media_id}" align="aligncenter" width="{new_width}"]{updatedImgTag} {new_alt_text}[/caption]'

                # Print the generated shortcode for debugging/example
                print(f"    Generated shortcode for {original_src}: {caption_shortcode}")

                # Replace the old img tag with the shortcode string directly
                # Convert the shortcode string to a NavigableString before replacing
                from bs4 import NavigableString
                img_tag.replace_with(NavigableString(caption_shortcode))


                images_replaced_count += 1
                print(f"    Replaced image tag for: {original_src} with caption shortcode.")
            elif original_src:
                print(f"    Image src not found in mapping, keeping original tag: {original_src}")
            else:
                print("    Found image tag with no src attribute.")


        # Get the modified HTML content as a string
        processed_html_content = str(soup)

        # Explicitly unescape the HTML entities for < and > within the shortcode
        processed_html_content = processed_html_content.replace('&lt;img', '<img')
        processed_html_content = processed_html_content.replace('/&gt;', '/>') # Handle the closing tag as well


        print(f"  Finished replacing image tags. {images_replaced_count} image tags replaced.")

        # Store the processed HTML content in a variable for the next steps
        # You might want to store this in a dictionary keyed by filename if processing multiple files
        if 'processed_html_contents' not in globals():
            processed_html_contents = {}
        processed_html_contents[html_filepath] = processed_html_content
        print(f"  Processed HTML content stored for {html_filepath}.")

    except FileNotFoundError:
        print(f"  Error: HTML file not found at {html_filepath}")
    except Exception as e:
        print(f"  An unexpected error occurred while processing {html_filepath}: {e}")

print("\n--- HTML Processing Summary ---")
if uploaded_filenames:
    print(f"Processed HTML file: {uploaded_filenames[0]}")
    print(f"Image tags replaced: {images_replaced_count}")
else:
    print("No HTML file processed.")

# You can now use the 'processed_html_contents' variable for later steps


--- Processing HTML and Replacing Image Tags ---
  Reading HTML file: Kệ-trung-tải-4-tầng-Tối-ưu-hóa-không-gian-theo-chiều-cao-tăng-gấp-đôi-hiệu-suất.html
  Found 6 image tags in the HTML.
    Generated shortcode for https://lh7-rt.googleusercontent.com/docsz/AD_4nXcuP4ohYa6hWXz8ElzVkbJHmv8IYPkzsz5MqRFpbxoJdOx5GD8B7_5Whcud-b3C9qe5xxPtKRry_O27HcVcKXOBxVSROm_tY8GIV1xhyEvuPIpOVoIxSo5TEkoBl2E6Iuk7djo3PA?key=5axpOQhltCUHo8WHcYAGCg: [caption id="attachment_47147" align="aligncenter" width="800"]<img class="aligncenter size-full wp-image-47147" title="" src="https://ngoncareer.com/wp-content/uploads/2025/09/ke-trung-tai-4-tang-17.png" alt="Kệ trung tải 4 tầng" width="800" height="600" /> Kệ trung tải 4 tầng[/caption]
    Replaced image tag for: https://lh7-rt.googleusercontent.com/docsz/AD_4nXcuP4ohYa6hWXz8ElzVkbJHmv8IYPkzsz5MqRFpbxoJdOx5GD8B7_5Whcud-b3C9qe5xxPtKRry_O27HcVcKXOBxVSROm_tY8GIV1xhyEvuPIpOVoIxSo5TEkoBl2E6Iuk7djo3PA?key=5axpOQhltCUHo8WHcYAGCg with caption shortcode.
    Generated 

## Process other Html elements

### Modify general setting

In [20]:
import re

def clean_html(output_html: str) -> str:
    """
    Cleans HTML content by removing specific styles, comments, tags,
    normalizing spacing, and unescaping certain characters.

    Args:
        output_html: The HTML content string to clean.

    Returns:
        The cleaned HTML content string.
    """
    # REMOVE STYLES from <img ... style="...">
    def _strip_img_style(m: re.Match) -> str:
        # remove only the style="..." within the matched <img ...> chunk
        return re.sub(r'\s*style="[^"]*"', '', m.group(0), flags=re.IGNORECASE)

    output_html = re.sub(
        r'<img\s+[^>]*?style="[^"]*"',
        _strip_img_style,
        output_html,
        flags=re.IGNORECASE
    )

    # REMOVE COMMENTS (the specific <div><p><a id="cmnt\d+">[...]</a><span>...</span></p></div> block)
    output_html = re.sub(
        r'<div[^>]*>\s*<p[^>]*>\s*<a[^>]*id="cmnt\d+"[^>]*>\[.*?\]</a>\s*<span[^>]*>.*?</span>\s*</p>\s*</div>',
        '',
        output_html,
        flags=re.IGNORECASE | re.DOTALL
    ).strip()

    # REMOVE <H1>
    output_html = re.sub(
        r'<h1[^>]*>.*?</h1>',
        ' ',
        output_html,
        flags=re.IGNORECASE | re.DOTALL
    ).strip()

    # REMOVE TITLE (p.class="title")
    output_html = re.sub(
        r'<p\b[^>]*\bclass=["\']title["\'][^>]*>[\s\S]*?</p>',
        '',
        output_html,
        flags=re.IGNORECASE
    )

    # TABLE MAX WIDTH (append width:100% to inline table style)
    output_html = re.sub(
        r'<table\s+[^>]*?style="([^"]*)"',
        r'<table style="\1;width: 100%;"',
        output_html,
        flags=re.IGNORECASE
    )

    # Remove <style> blocks, normalize &nbsp;, collapse whitespace
    output_html = re.sub(
        r'<style.*?</style>',
        '',
        output_html,
        flags=re.IGNORECASE | re.DOTALL
    )
    output_html = re.sub(r'&nbsp;', ' ', output_html, flags=re.IGNORECASE)
    output_html = re.sub(r'\s+', ' ', output_html).strip()

    # REMOVE FIRST ROW SPACE (leading &nbsp; at start of doc)
    output_html = re.sub(r'^(\s*&nbsp;\s*)+', '', output_html, flags=re.IGNORECASE)

    # REMOVE google.com redirector from all links
    output_html = re.sub(
        r'<a\s+[^>]*?href="https://www\.google\.com/url\?q=([^&"]*)[^"]*"',
        r'<a href="\1"',
        output_html,
        flags=re.IGNORECASE
    )

    # REMOVE FONT FAMILY / SIZE inline CSS
    output_html = re.sub(
        r'font-(family|size):\s*[^;"\']+;?',
        '',
        output_html,
        flags=re.IGNORECASE
    )

    # FIX ITALIC FONT-STYLE => move to <em> and keep remaining styles
    italic_pattern = re.compile(
        r'<([^>]+)\s+([^>]*)style="([^"]*?)font-style:\s*italic;?([^"]*?)"([^>]*)>(.*?)</\1>',
        flags=re.IGNORECASE | re.DOTALL
    )

    def _italic_repl(m: re.Match) -> str:
        tag, before_style, pre_style, post_style, after_style, content = m.groups()
        pre_style = pre_style.strip()
        post_style = post_style.strip()
        new_style = '; '.join(s for s in (pre_style, post_style) if s)
        if new_style:
            new_tag_open = f'<{tag} {before_style}style="{new_style}"{after_style}>'
        else:
            new_tag_open = f'<{tag} {before_style}{after_style}>'
        return f'{new_tag_open}<em>{content}</{tag}>'

    output_html = italic_pattern.sub(_italic_repl, output_html)

    return output_html

# Code to apply the clean_html function to processed_html_contents
if 'processed_html_contents' in globals() and processed_html_contents and 'uploaded_filenames' in globals() and uploaded_filenames:
    print("\n--- Applying clean_html to processed_html_contents ---")
    cleaned_count = 0
    for html_filepath in uploaded_filenames: # Iterate through processed files
        if html_filepath in processed_html_contents:
            original_content = processed_html_contents[html_filepath]
            cleaned_content = clean_html(original_content)
            processed_html_contents[html_filepath] = cleaned_content # Update with cleaned content
            cleaned_count += 1
            print(f"  Cleaned HTML content for {html_filepath}")
        else:
            print(f"  Processed HTML content not found for {html_filepath}. Skipping.")

    print(f"Finished applying clean_html to {cleaned_count} processed HTML contents.")
elif 'processed_html_contents' not in globals() or not processed_html_contents:
    print("\n--- processed_html_contents not available. Skipping HTML cleaning. ---")
elif 'uploaded_filenames' not in globals() or not uploaded_filenames:
     print("\n--- uploaded_filenames not available. Skipping HTML cleaning. ---")


--- Applying clean_html to processed_html_contents ---
  Cleaned HTML content for Kệ-trung-tải-4-tầng-Tối-ưu-hóa-không-gian-theo-chiều-cao-tăng-gấp-đôi-hiệu-suất.html
Finished applying clean_html to 1 processed HTML contents.


### Modify <p tags

In [21]:
import re

def normalize_p_tags(output_html: str) -> str:
    # 3.1 REPLACE ALL <P TAGS>
    # Normalize ALL <p> tags
    def _normalize_open_p(m: re.Match) -> str:
        open_p = m.group(0)
        has_center  = bool(re.search(r'text-align\s*:\s*center',  open_p, re.IGNORECASE)) or \
                      bool(re.search(r'\balign\s*=\s*["\']?\s*center\s*["\']?', open_p, re.IGNORECASE))
        has_justify = bool(re.search(r'text-align\s*:\s*justify', open_p, re.IGNORECASE)) or \
                      bool(re.search(r'\balign\s*=\s*["\']?\s*justify\s*["\']?', open_p, re.IGNORECASE))
        has_right   = bool(re.search(r'text-align\s*:\s*right',   open_p, re.IGNORECASE)) or \
                      bool(re.search(r'\balign\s*=\s*["\']?\s*right\s*["\']?', open_p, re.IGNORECASE))

        if has_center:
            return '<p dir="ltr" style="text-align: center;">'
        if has_justify:
            return '<p dir="ltr" style="text-align: justify;">'
        if has_right:
            return '<p dir="ltr" style="text-align: right;">'
        return '<p dir="ltr" style="text-align: justify;">'

    output_html = re.sub(r'<p\b[^>]*>', _normalize_open_p, output_html, flags=re.IGNORECASE)

    # Ensure any <p> that contains an <img> gets centered
    def _center_p_with_img(m: re.Match) -> str:
        attrs, inner_html = m.group(1), m.group(2)
        # Strip existing style="" from the <p> tag's attributes
        clean_attrs = re.sub(r'\s*style="[^"]*"', '', attrs, flags=re.IGNORECASE)
        # Rebuild the <p> with text-align:center, keeping other attributes
        # (Note: mirrors original JS, which does not close the <span>.)
        return f'<p{clean_attrs} style="text-align: center;"><span style="font-weight: 400">{inner_html}</p>'

    output_html = re.sub(
        r'<p\b([^>]*)>((?:(?!</p>).)*?<img\b[^>]*>[\s\S]*?)</p>',
        _center_p_with_img,
        output_html,
        flags=re.IGNORECASE | re.DOTALL
    )

    return output_html

# Code to apply the normalize_p_tags function to processed_html_contents
if 'processed_html_contents' in globals() and processed_html_contents and 'uploaded_filenames' in globals() and uploaded_filenames:
    print("\n--- Applying normalize_p_tags to processed_html_contents ---")
    normalized_count = 0
    for html_filepath in uploaded_filenames: # Iterate through processed files
        if html_filepath in processed_html_contents:
            original_content = processed_html_contents[html_filepath]
            normalized_content = normalize_p_tags(original_content)
            processed_html_contents[html_filepath] = normalized_content # Update with normalized content
            normalized_count += 1
            print(f"  Normalized <p> tags in HTML content for {html_filepath}")
        else:
            print(f"  Processed HTML content not found for {html_filepath}. Skipping.")

    print(f"Finished applying normalize_p_tags to {normalized_count} processed HTML contents.")
elif 'processed_html_contents' not in globals() or not processed_html_contents:
    print("\n--- processed_html_contents not available. Skipping <p> tag normalization. ---")
elif 'uploaded_filenames' not in globals() or not uploaded_filenames:
     print("\n--- uploaded_filenames not available. Skipping <p> tag normalization. ---")


--- Applying normalize_p_tags to processed_html_contents ---
  Normalized <p> tags in HTML content for Kệ-trung-tải-4-tầng-Tối-ưu-hóa-không-gian-theo-chiều-cao-tăng-gấp-đôi-hiệu-suất.html
Finished applying normalize_p_tags to 1 processed HTML contents.


### Modify span, strong tags

In [22]:
import re

def span_bold_to_strong(output_html: str) -> str:
    """
    Converts <span> tags with font-weight: bold or 700 style to <strong> tags.

    Args:
        output_html: The HTML content string to process.

    Returns:
        The HTML content string with bold spans converted to strong tags.
    """
    pattern = re.compile(
        r"""<span\b[^>]*\bstyle\s*=\s*(['"])           # style=" or style='
            (?:(?!\1).)*                                # anything up to the same quote
            \bfont-weight\s*:\s*(?:bold|700)\b         # font-weight: bold/700
            (?:(?!\1).)*\1                              # rest of style until the same quote
            [^>]*>                                      # end of opening <span ...>
            ([\s\S]*?)                                  # capture inner content
            </span>                                     # closing tag
        """,
        flags=re.IGNORECASE | re.DOTALL | re.VERBOSE
    )
    return pattern.sub(r'<strong>\2</strong>', output_html)

def strip_span_tags(output_html: str) -> str:
    """
    Removes all <span> tags while keeping their inner content.

    Args:
        output_html: The HTML content string to process.

    Returns:
        The HTML content string with <span> tags removed.
    """
    # Remove any opening <span ...> but keep inner content
    output_html = re.sub(r'<span\b[^>]*>', '', output_html, flags=re.IGNORECASE)
    # Remove any closing </span>
    output_html = re.sub(r'</span>', '', output_html, flags=re.IGNORECASE)
    return output_html

# Code to apply the span and strong tag processing functions to processed_html_contents
if 'processed_html_contents' in globals() and processed_html_contents and 'uploaded_filenames' in globals() and uploaded_filenames:
    print("\n--- Applying span and strong tag processing to processed_html_contents ---")
    processed_count = 0
    for html_filepath in uploaded_filenames: # Iterate through processed files
        if html_filepath in processed_html_contents:
            original_content = processed_html_contents[html_filepath]

            # Apply span_bold_to_strong
            content_after_bold_conversion = span_bold_to_strong(original_content)

            # Apply strip_span_tags
            final_processed_content = strip_span_tags(content_after_bold_conversion)

            processed_html_contents[html_filepath] = final_processed_content # Update with final processed content
            processed_count += 1
            print(f"  Processed span and strong tags in HTML content for {html_filepath}")
        else:
            print(f"  Processed HTML content not found for {html_filepath}. Skipping.")

    print(f"Finished applying span and strong tag processing to {processed_count} processed HTML contents.")
elif 'processed_html_contents' not in globals() or not processed_html_contents:
    print("\n--- processed_html_contents not available. Skipping span and strong tag processing. ---")
elif 'uploaded_filenames' not in globals() or not uploaded_filenames:
     print("\n--- uploaded_filenames not available. Skipping span and strong tag processing. ---")


--- Applying span and strong tag processing to processed_html_contents ---
  Processed span and strong tags in HTML content for Kệ-trung-tải-4-tầng-Tối-ưu-hóa-không-gian-theo-chiều-cao-tăng-gấp-đôi-hiệu-suất.html
Finished applying span and strong tag processing to 1 processed HTML contents.


###Modify headings

###Modify table

### Modify unordered and ordered list

### Return the processed_html file

In [23]:
# Display and export the processed HTML content
import os

if 'processed_html_contents' in globals() and uploaded_filenames:
    html_filepath = uploaded_filenames[0]
    if html_filepath in processed_html_contents:
        processed_html_content = processed_html_contents[html_filepath]

        print("\n--- Processed HTML Content (Snippet) ---")
        # Print the type and a snippet of the processed HTML content
        print(f"Type of processed_html_content: {type(processed_html_content)}")
        print(processed_html_content[:1000]) # Print the first 1000 characters

        print("\n--- Full Processed HTML Content ---")
        print(processed_html_content)


        # Determine the output filename for the processed HTML
        base_name, ext = os.path.splitext(html_filepath)
        output_html_filepath = f"{base_name}_processed{ext}"

        # Export the processed HTML content to a new file
        try:
            with open(output_html_filepath, 'w', encoding='utf-8') as f:
                f.write(processed_html_content)
            print(f"\nProcessed HTML content exported to: {output_html_filepath}")
        except IOError as e:
            print(f"\nError exporting processed HTML to {output_html_filepath}: {e}")

    else:
        print(f"Processed HTML content not found for {html_filepath}.")
else:
    print("Processed HTML content not available. Please ensure the HTML processing step was executed successfully.")


--- Processed HTML Content (Snippet) ---
Type of processed_html_content: <class 'str'>
<html><head><meta content="text/html; charset=utf-8" http-equiv="content-type"/></head><body class="doc-content" style="background-color:#ffffff;max-width:451.4pt;padding:72pt 72pt 72pt 72pt"> <p dir="ltr" style="text-align: justify;">Khi chi phí thuê kho ngày càng tăng, việc mở rộng diện tích lưu trữ trở nên tốn kém và khó khả thi, buộc doanh nghiệp phải tận dụng tối đa không gian hiện có. Hệ quả là không gian lưu trữ trở nên hạn chế, gây khó khăn trong quản lý hàng hóa.</p><p dir="ltr" style="text-align: justify;"><em>Trong bối cảnh này, <strong><em>kệ trung tải 4 tầng</strong><em> là giải pháp tối ưu để vừa tận dụng triệt để chiều cao vừa tăng sức chứa kho bãi. Bài viết dưới đây sẽ giải thích chi tiết các lợi ích khi sử dụng kệ, đồng thời cung cấp báo giá tham khảo mới nhất.</p><p dir="ltr" style="text-align: center;">[caption id="attachment_47147" align="aligncenter" width="800"]<img class="alig

# 5. Updating wordpress post